In [1]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("ATM_Analytics_Jharkhand.csv")

In [6]:
print(df.head)

<bound method NDFrame.head of         Year Month Bank      State        City  Effective_Days     ATM_Type  \
0       2023   Feb  SBI  Jharkhand      Ranchi              28       Onsite   
1       2023   Mar  SBI  Jharkhand     Deoghar              29  Brown Label   
2       2024   Apr  SBI  Jharkhand     Dhanbad              30      Offsite   
3       2023   Nov  SBI  Jharkhand     Deoghar              27  Brown Label   
4       2024   Feb  SBI  Jharkhand  Hazaribagh              25       Onsite   
...      ...   ...  ...        ...         ...             ...          ...   
299995  2024   Oct  SBI  Jharkhand     Deoghar              30      Offsite   
299996  2023   Feb  SBI  Jharkhand     Deoghar              29      Offsite   
299997  2023   Jan  SBI  Jharkhand     Deoghar              26  Brown Label   
299998  2024   Mar  SBI  Jharkhand      Bokaro              27      Offsite   
299999  2024   Jul  SBI  Jharkhand  Hazaribagh              29       Onsite   

             ATM_ID Q

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Year                  300000 non-null  int64  
 1   Month                 300000 non-null  object 
 2   Bank                  300000 non-null  object 
 3   State                 300000 non-null  object 
 4   City                  300000 non-null  object 
 5   Effective_Days        300000 non-null  int64  
 6   ATM_Type              300000 non-null  object 
 7   ATM_ID                300000 non-null  object 
 8   Quarter               300000 non-null  object 
 9   Fin_Txn               300000 non-null  int64  
 10  Non_Fin_Txn           300000 non-null  float64
 11  Monthly_Txn           300000 non-null  float64
 12  Monthly_Revenue       300000 non-null  float64
 13  MHA_Revenue           300000 non-null  float64
 14  ATM_Revenue_Total     300000 non-null  float64
 15  

In [7]:
df.dtypes

Year                      int64
Month                    object
Bank                     object
State                    object
City                     object
Effective_Days            int64
ATM_Type                 object
ATM_ID                   object
Quarter                  object
Fin_Txn                   int64
Non_Fin_Txn             float64
Monthly_Txn             float64
Monthly_Revenue         float64
MHA_Revenue             float64
ATM_Revenue_Total       float64
Rent                      int64
Electricity_Bill        float64
ATM_AMC                   int64
UPS_AMC                   int64
VSAT_AMC                  int64
Spare_Rep_SLM             int64
Spare_Rep_AC              int64
Spare_Rep_UPS             int64
Compensation              int64
Penalty                   int64
Housekeeping              int64
Security                  int64
Insurance                 int64
Total_Cost              float64
Gross_Profit            float64
Gross_Profit_%          float64
Txn_Rang

# Duplicate Check

In [9]:
df.duplicated().sum()
df["ATM_ID"].duplicated().sum()

np.int64(0)

# Monthly Transaction check

In [13]:
Txn_chk = (df["Monthly_Txn"] != df["Fin_Txn"] + df["Non_Fin_Txn"]).sum()

In [14]:
Txn_chk

np.int64(146293)

In [25]:
df["Monthly_Txn"] = df["Fin_Txn"] + df["Non_Fin_Txn"]

In [26]:
df["Monthly_Txn"] = df["Non_Fin_Txn"].round(0)

In [27]:
df["Monthly_Txn"] = df["Non_Fin_Txn"].astype(int)

In [28]:
df["Monthly_Txn"] = df["Fin_Txn"] + df["Non_Fin_Txn"]

In [29]:
Txn_chk = (df["Monthly_Txn"] != df["Fin_Txn"] + df["Non_Fin_Txn"]).sum()

In [30]:
Txn_chk

np.int64(0)

# Revenue Check

In [31]:
Rev_chk = (df["ATM_Revenue_Total"] != df["Monthly_Revenue"] + df["MHA_Revenue"]).sum()
Rev_chk

np.int64(245107)

In [32]:
df["ATM_Revenue_Total"] = df["Monthly_Revenue"] + df["MHA_Revenue"]

In [33]:
Rev_chk = (df["ATM_Revenue_Total"] != df["Monthly_Revenue"] + df["MHA_Revenue"]).sum()
Rev_chk

np.int64(0)

# Profit Validation

In [34]:
Profit_chk = (df["Gross_Profit"] != df["ATM_Revenue_Total"] + df["Total_Cost"]).sum()
Profit_chk

np.int64(300000)

In [35]:
df["Gross_Profit"] = df["ATM_Revenue_Total"] + df["Total_Cost"]

In [37]:
Profit_chk = (df["Gross_Profit"] != df["ATM_Revenue_Total"] + df["Total_Cost"]).sum()
Profit_chk

np.int64(0)

# -ve Profit Chk

In [38]:
df[df["Gross_Profit_%"] < -100].shape

(60302, 37)

In [40]:
def profit_flag(x):
    if x < 0:
        return "Loss"
    elif x <= 20:
        return "Low Profit"
    else:
        return "High Profit"
df["Profit_Category"] = df["Gross_Profit_%"].apply(profit_flag)

# Recalculating Profit % chk 

In [41]:
df["Gross_Profit_%"] = (df["Gross_Profit"] / df["ATM_Revenue_Total"]) * 100
df["Gross_Profit_%"] = df["Gross_Profit_%"].round(2)

In [42]:
df["Gross_Profit_%"].head()

0    286.55
1    224.81
2    587.95
3    228.34
4    210.02
Name: Gross_Profit_%, dtype: float64

# Uptime Sanity Chk

In [43]:
df["Uptime"].describe()

count    300000.000000
mean         95.003162
std           2.882295
min          90.000013
25%          92.507189
50%          94.999659
75%          97.495456
max          99.999997
Name: Uptime, dtype: float64

In [44]:
df["Uptime"] = df["Uptime"].round(2)

In [45]:
df["Uptime"].head()

0    99.54
1    95.79
2    96.46
3    93.52
4    96.81
Name: Uptime, dtype: float64

# Category Consistency Chk

In [48]:
df["Txn_Range_Current"].unique()
df["Margin_Status"].unique()
df["Revenue_Performance"].unique()

array(['Poor', 'Good'], dtype=object)

In [49]:
df["Txn_Range_Current"].unique()
df["Margin_Status"].unique()

array(['Below 0%', 'Above 0%'], dtype=object)

# Reordering Month 

In [50]:
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

In [51]:
df["Month"].head(20)

0     Feb
1     Mar
2     Apr
3     Nov
4     Feb
5     Jul
6     Dec
7     Jan
8     Mar
9     Apr
10    Dec
11    Jul
12    Jul
13    Dec
14    Apr
15    Feb
16    Nov
17    Apr
18    Jun
19    Apr
Name: Month, dtype: category
Categories (12, object): ['Jan' < 'Feb' < 'Mar' < 'Apr' ... 'Sep' < 'Oct' < 'Nov' < 'Dec']

# Rounding off to 2 Decimal Places

In [53]:
df["Non_Fin_Txn"] = df["Non_Fin_Txn"].round(2)
df["Monthly_Txn"] = df["Monthly_Txn"].round(2)
df["Monthly_Revenue"] = df["Monthly_Revenue"].round(2)
df["MHA_Revenue"] = df["MHA_Revenue"].round(2)
df["ATM_Revenue_Total"] = df["ATM_Revenue_Total"].round(2)
df["Electricity_Bill"] = df["Electricity_Bill"].round(2)
df["Total_Cost"] = df["Total_Cost"].round(2)
df["Gross_Profit"] = df["Gross_Profit"].round(2)

# Export

In [55]:
df.to_csv("ATM_Analysis_final_cleaned.csv", index=False)

In [ ]:
df.to_csv("clean_data.csv", index=False)